In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns;

In [ ]:
um = pd.read_csv("TWO_CENTURIES_OF_UM_RACES")

In [ ]:
um.head()

In [ ]:
um.tail()

In [ ]:
um.dtypes

In [ ]:
um.shape

In [ ]:
um.describe()

In [ ]:
um.info()

In [ ]:
um.isna().sum()

In [ ]:
#show dat with null values
um[um.isnull().any(axis=1)]

In [ ]:
#change year dtype to int
um['Year of event'] = um['Year of event'].astype(int)

In [ ]:
um['Event dates'] = pd.to_datetime(um['Event dates'], errors='coerce')

In [ ]:
# remove 'h' from Athlete performance and convert to hours and minutes
um['Athlete performance'] = um['Athlete performance'].str.replace('h', '')
um['Athlete performance'] = pd.to_datetime(um['Athlete performance'], format='%H:%M', errors='coerce')

In [ ]:
um['Athlete year of birth'] = pd.to_numeric(um['Athlete year of birth'], errors='coerce').astype('Int64')

In [ ]:
um['Athlete average speed'] = pd.to_numeric(um['Athlete average speed'], errors='coerce')

In [ ]:
um.dtypes

In [ ]:
#all the unique values in the column 'Event distance/length'
um['Event distance/length'].unique().all


In [ ]:
#see count of athletes with '50km', '50mi','100km', '100mi','102km', '102mi','201km'
um[um['Event distance/length'].isin(['50km', '50mi','100km', '100mi','102km', '102mi','201km'])] 

In [ ]:
um[um['Event distance/length'].isin(['6h', '12h', '24h'])]

In [ ]:
um[um['Event distance/length'].isin(['6h', '12h', '24h'])].isna().sum()

In [ ]:
um['Year of event'].unique()

In [ ]:
#most repeated event names along with their counts greater than 1
um['Event name'].value_counts() [um['Event name'].value_counts()>1000]

In [ ]:
excluded = ['6h', '12h', '24h', '50km', '50mi', '100km', '100mi', '102km', '102mi', '201km']
~(um['Event distance/length'].isin(excluded)).sum()


In [ ]:
# Example: Separating '50km', '100mi', '24h' into value and unit
um[['distance_value', 'distance_unit']] = um['Event distance/length'].str.extract(r'^([\d\.]+)\s*([a-zA-Z]+)$')

# Convert the extracted number column from text to float
um['distance_value'] = pd.to_numeric(um['distance_value'])

In [ ]:
um[ 'distance_unit'].unique()

In [ ]:
#change km,k,K,Km to kms and mi,mile,m,Miles,miles to miles
um['distance_unit'] = um['distance_unit'].str.replace(r'^(km|k|K|Km)$', 'kms', regex=True)
um['distance_unit'] = um['distance_unit'].str.replace(r'^(mi|mile|m|Miles|miles|Mile)$', 'miles', regex=True)

In [ ]:
#changing milesto km
um['Distance in km'] = np.where(um['distance_unit']=='kms', um['distance_value'], um['distance_value'] * 1.6093)

In [ ]:
um[ 'distance_unit'].unique()

In [ ]:
#check the count of ('h','d')
check = ('h','d', np.nan)
# count occurrences of the units in 'check' ('h' and 'd')
um.loc[um['distance_unit'].isin(check), 'distance_unit'].value_counts().reindex(list(check), fill_value=0)

In [ ]:
um[um['distance_unit']== 'd']

In [ ]:
um.drop(columns=['Event distance/length'], inplace=True)
um

In [ ]:
#find athlete age
um['Athlete age'] = um['Year of event'] - um['Athlete year of birth']
um.sample(3)

In [ ]:
um['Event Country'] = um['Event name'].str.extract(r"\(([A-Z]{3})\)$")
um['Event Country'].unique()

In [ ]:
um['Athlete performance'].unique()

In [ ]:
um['Athlete performance'].isna().sum()

In [ ]:
um.drop(columns=['Athlete performance'], inplace=True)
um.info()

In [ ]:
#create a column named decade according to the year of event
um['Decade'] = (um['Year of event'] // 10) * 10
um

In [ ]:
um.drop(columns=['Event dates', 'Athlete year of birth'], inplace=True)
um.info()

In [ ]:
um.duplicated().sum()

In [ ]:
um.drop_duplicates(inplace=True)
um.duplicated().sum()

In [ ]:
# Count rows containing at least one null value
um.isna().any(axis=1).sum()


In [ ]:
#drop null values in the dataset
um.dropna(inplace=True)

In [ ]:
um.drop(['distance_value'],axis=1, inplace=True)

In [ ]:
um.info()

In [ ]:
#rearrange the columns in the dataset
ds = um[['Athlete ID', 'Athlete age','Athlete gender','Athlete club','Athlete country','Athlete age category','Event name', 'Event Country', 'Year of event', 'Decade', 'Distance in km', 'distance_unit', 'Athlete average speed']]
ds

In [ ]:
athletes_by_country = (
    ds.dropna(subset=['Athlete country'])
      .groupby('Athlete country')['Athlete ID']
      .nunique()
      .sort_values(ascending=False)
      .head(20)
      .sort_values()
)

plt.figure(figsize=(12, 8))
athletes_by_country.plot(kind='barh', color='steelblue')
plt.xlabel('Number of unique athletes')
plt.ylabel('Athlete country')
plt.title('Top 20 Countries by Number of Athletes')
plt.tight_layout()
plt.show()